# Manual Code Features Experiments

In [6]:
import numpy as np
import pandas as pd

In [7]:
def load_data(csv_path: str) -> pd.DataFrame:
    """Load data CSV file into a DataFrame."""
    return pd.read_csv(csv_path)

In [8]:
def parse_filename(fn: str):
    """
    Splits a filename into (executable, option, compiler).
    """
    # Map from filename‐stem prefixes to your canonical program names:
    PREFIX_TO_PROGRAM = {
        "parse_xml": "cparserXML",
        "example":   "csimpleJSONparser",
        "http_parser": "benoitc_HTTP",
        "cjson": "cJSON",
        "picohttpparser": "picohttpparser",
        "calc": "yacc_calculator",
        "elf-parser": "elf_parser",
        "network_analyzer": "network_packet_analyzer",
        "packcc": "packcc",
        "pcap_parser": "pcap_parser",
    }

    # Compilers
    KNOWN_COMPILERS = {'gcc', 'clang'}
    # Optimization tokens
    OPT_TOKEN_MAP = {
        '0': 'O0', '1': 'O1', '2': 'O2', '3': 'O3',
        's': 'Os', 'fast': 'Ofast',
        'O0': 'O0', 'O1': 'O1', 'O2': 'O2', 'O3': 'O3',
        'Os': 'Os', 'Ofast': 'Ofast',
    }
    # strip extension
    stem = fn.rsplit(".", 1)[0]
    
    # find which prefix it starts with
    # take the longest matching key so that "parse_xml" beats "par" if both existed
    candidates = [p for p in PREFIX_TO_PROGRAM if stem.startswith(p)]
    if candidates:
        # pick the longest matching key
        best = max(candidates, key=len)
        program = PREFIX_TO_PROGRAM[best]
        remainder = stem[len(best):].lstrip("_")
    else:
        # fallback: use everything before first underscore
        parts = stem.split("_", 1)
        program = parts[0]
        remainder = parts[1] if len(parts) > 1 else ""
    
    # split off settings tokens
    settings = remainder.split("_") if remainder else []
    
    # compiler: first known compiler in settings, else default gcc
    compiler = next((s for s in settings if s in KNOWN_COMPILERS), "gcc")
    
    # optimization: map tokens via OPT_TOKEN_MAP or keep digits
    opts = []
    for s in settings:
        if s in OPT_TOKEN_MAP:
            opts.append(OPT_TOKEN_MAP[s])
        elif s.isdigit():
            opts.append(s)
    option = "_".join(opts) if opts else "default"
    
    return program, option, compiler

def enrich_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'program', 'option', 'compiler' columns to a DataFrame
    with an existing 'file_name' column.
    """
    parsed = df["file_name"].apply(parse_filename)
    df = df.copy()
    df[["program", "option", "compiler"]] = pd.DataFrame(parsed.tolist(), index=df.index)
    # Remove the 'filename' column as it's no longer needed
    df.drop(columns=["file_name"], inplace=True)
    return df

In [9]:
# Path to the CSV file
path = "../code_counting_features/outputs/combined_extracted.csv"
# Load the data
df = load_data(path)

print(f"Loaded dataframe with {len(df)} rows (=functions).")
print(f"Columns in DataFrame: {df.columns.tolist()}\n")
print(f"Unique executables (filenames): {df['file_name'].unique().tolist()}")
df.head()

Loaded dataframe with 7898 rows (=functions).
Columns in DataFrame: ['file_name', 'name', 'address', 'label', 'num_blocks', 'success', 'error_message', 'vex_ir', 'decompiled_c']

Unique executables (filenames): ['parse_xml', 'parse_xml_32', 'parse_xml_clang', 'parse_xml_32_clang', 'parse_xml_0', 'parse_xml_0_32', 'parse_xml_0_clang', 'parse_xml_0_32_clang', 'parse_xml_s', 'parse_xml_s_32', 'parse_xml_s_clang', 'parse_xml_s_32_clang', 'parse_xml_3', 'parse_xml_3_32', 'parse_xml_3_clang', 'parse_xml_3_32_clang', 'parse_xml_1', 'parse_xml_1_32', 'parse_xml_1_clang', 'parse_xml_1_32_clang', 'parse_xml_2', 'parse_xml_2_32', 'parse_xml_2_clang', 'parse_xml_2_32_clang', 'parse_xml_fast', 'parse_xml_fast_32', 'parse_xml_fast_clang', 'parse_xml_fast_32_clang', 'picohttpparser', 'picohttpparser_32', 'picohttpparser_clang', 'picohttpparser_32_clang', 'picohttpparser_0', 'picohttpparser_0_32', 'picohttpparser_0_clang', 'picohttpparser_0_32_clang', 'picohttpparser_s', 'picohttpparser_s_32', 'picoht

,file_name,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c
0,parse_xml,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...
1,parse_xml,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...
2,parse_xml,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...
3,parse_xml,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...
4,parse_xml,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...


In [10]:
# enrich the DataFrame with program, option, compiler columns
df = enrich_df(df) 
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc


In [11]:
import re
import numpy as np
import pandas as pd

In [12]:
# --- Regex & helpers ---
HEX_RE = re.compile(r'0x[0-9a-fA-F]+')
BLOCK_HEADER_RE = re.compile(r'---\s*BLOCK\s*(0x[0-9a-fA-F]+)\s*---')
SWITCH_RE = re.compile(r'\bswitch\b', re.IGNORECASE)
CASE_RE = re.compile(r'\bcase\b', re.IGNORECASE)
IF_RE = re.compile(r'\bif\s*\(', re.IGNORECASE)
LOOP_RE = re.compile(r'\b(for|while)\s*\(|\bdo\s*\{', re.IGNORECASE)
GOTO_RE = re.compile(r'\bgoto\b', re.IGNORECASE)

In [13]:
def _safe_str(s):
    return "" if s is None or (isinstance(s, float) and np.isnan(s)) else str(s)

In [14]:
# 1) switch
def extract_switch(decompiled_c, vex_ir, mode="both"):
    d, v = _safe_str(decompiled_c), _safe_str(vex_ir)
    if mode == "decompiled":
        return int(bool(SWITCH_RE.search(d) or CASE_RE.search(d)))
    if mode == "vex":
        return int(bool(re.search(r'jumptable|jump table|indirect_jump', v, re.IGNORECASE)))
    return int(bool(SWITCH_RE.search(d) or CASE_RE.search(d) or
                    re.search(r'jumptable|jump table|indirect_jump', v, re.IGNORECASE)))

# 2) switch_loop
def extract_switch_loop(decompiled_c, vex_ir, mode="both"):
    sw = extract_switch(decompiled_c, vex_ir, mode)
    d, v = _safe_str(decompiled_c), _safe_str(vex_ir)

    has_loop_dec = bool(LOOP_RE.search(d))
    headers = BLOCK_HEADER_RE.findall(v)
    has_loop_vex = bool(GOTO_RE.search(v)) or (len(headers) > 1)

    if mode == "decompiled": has_loop = has_loop_dec
    elif mode == "vex": has_loop = has_loop_vex
    else: has_loop = has_loop_dec or has_loop_vex
    return int(sw and has_loop)

# 3) br_fact
def extract_br_fact(decompiled_c, vex_ir, mode="both"):
    d, v = _safe_str(decompiled_c), _safe_str(vex_ir)
    candidates = []
    if mode in ("decompiled", "both"):
        if IF_RE.search(d): candidates.append(2)
        else_if_count = len(re.findall(r'\belse\s+if\s*\(', d, re.IGNORECASE))
        if else_if_count: candidates.append(2 + else_if_count)
        if SWITCH_RE.search(d):
            cases = len(CASE_RE.findall(d))
            candidates.append(cases if cases else 4)
    if mode in ("vex", "both") and not candidates:
        if re.search(r'jumptable|indirect_jump', v, re.IGNORECASE): candidates.append(4)
    return max(candidates) if candidates else 0

# 4) in_edges
def extract_in_edges(vex_ir, mode="both"):
    v = _safe_str(vex_ir)
    if mode == "decompiled": return 0
    counts = {}
    for h in HEX_RE.findall(v):
        counts[h] = counts.get(h, 0) + 1
    return max(counts.values()) if counts else 0

# 5) bb_cnt
def extract_bb_cnt(num_blocks, vex_ir, mode="both"):
    try:
        if num_blocks and int(num_blocks) > 0:
            return int(num_blocks)
    except: pass
    return len(BLOCK_HEADER_RE.findall(_safe_str(vex_ir)))

# 6) call_cnt
def extract_call_cnt(func_name, all_decompiled, this_decompiled, mode="both"):
    if mode == "vex": return 0
    shortname = func_name.split('.')[-1]
    patt = re.compile(r'\b' + re.escape(shortname) + r'\s*\(')
    cnt = 0
    for body in all_decompiled:
        if body != this_decompiled and patt.search(body):
            cnt += 1
    return cnt

In [15]:
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc


In [ ]:
all_decompiled = df["decompiled_c"].fillna("").tolist()

df["switch_parc3"]      = df.apply(lambda r: extract_switch(decompiled_c=r["decompiled_c"], vex_ir=r["vex_ir"], mode="both"), axis=1)
df["switch_loop_parc3"] = df.apply(lambda r: extract_switch_loop(decompiled_c=r["decompiled_c"], vex_ir=r["vex_ir"], mode="both"), axis=1)
df["br_fact_parc3"]     = df.apply(lambda r: extract_br_fact(decompiled_c=r["decompiled_c"], vex_ir=r["vex_ir"], mode="both"), axis=1)
df["in_edges_parc3"]    = df.apply(lambda r: extract_in_edges(vex_ir=r["vex_ir"], mode="both"), axis=1)
df["bb_cnt_parc3"]      = df.apply(lambda r: extract_bb_cnt(num_blocks=r["num_blocks"], vex_ir=r["vex_ir"], mode="both"), axis=1)
df["call_cnt_parc3"]    = df.apply(
    lambda r: extract_call_cnt(func_name=r["name"], all_decompiled=all_decompiled, this_decompiled=r["decompiled_c"],  mode="both"), axis=1
)

In [ ]:
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46


In [16]:
output_csv = "../code_counting_features/outputs/extracted_datas_with_parc3.csv"

# Save with UTF-8 encoding and quotes (safer for long VEX/Decompiled strings)
#df.to_csv(output_csv, index=False, encoding="utf-8", quoting=1)  # quoting=1 = csv.QUOTE_ALL
#print(f"[+] Saved dataframe with parc3 features to: {output_csv}")

# Verify loading
df = load_data(output_csv)
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46


## Manual Score from paper PIE + using the same weights as in the paper

$score = \sum_{f \in Features}{} w_{f} \frac{ x_{f} - min(X_{f}) }{max(X_{f}) - min(X_{f})}$

In [17]:
def compute_parc3_score(df: pd.DataFrame, weights, threshold) -> pd.DataFrame:
    df = df.copy()
    score = 0
    
    for feature, w in weights.items():
        min_val = df[feature].min()
        max_val = df[feature].max()
        if max_val > min_val:
            norm = (df[feature] - min_val) / (max_val - min_val)
        else:
            norm = 0  # constant feature
        score += w * norm
    
    df["parc3_score"] = score
    df["parc3_pred"] = (df["parc3_score"] > threshold).astype(int)
    return df

In [18]:
# given weights
weights = {
    "switch_loop_parc3": 0.695,
    "br_fact_parc3": 0.502,
    "in_edges_parc3": 0.364,
    "bb_cnt_parc3": 0.413,
    "call_cnt_parc3": 0.134,
    "switch_parc3": 0.685,
}

T = 0.247  # threshold

df = compute_parc3_score(df, weights=weights, threshold=T)
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3,parc3_score,parc3_pred
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384,0.144967,0
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259,0.118277,0
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216,0.107769,0
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121,0.026668,0
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46,0.072437,0


In [21]:
def compute_recall_per_file(df: pd.DataFrame) -> pd.DataFrame:
    recalls = []
    
    for file, group in df.groupby("program"):
        tp = ((group["label"] == 1) & (group["parc3_pred"] == 1)).sum()
        fn = ((group["label"] == 1) & (group["parc3_pred"] == 0)).sum()
        
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        
        recalls.append({"file_name": file, "recall": recall})
    
    return pd.DataFrame(recalls)

In [22]:
recall_df = compute_recall_per_file(df)
print(recall_df)

                 file_name    recall
0             benoitc_HTTP  0.115789
1                    cJSON  0.015385
2               cparserXML  0.014706
3        csimpleJSONparser  0.021739
4               elf_parser  0.039301
5  network_packet_analyzer  0.000000
6                   packcc  0.130824
7              pcap_parser  0.000000
8           picohttpparser  0.062874
9          yacc_calculator  0.000000


In [1]:
import numpy as np

values = [0.115789, 0.015385, 0.014706, 0.021739, 0.039301, 0.000000, 0.130824, 0.000000, 0.062874, 0.000000]

print(np.mean(values), np.std(values))

0.0400618 0.045711913724542315


## Feature Normalization + XGBoost

In [57]:
output_csv = "../code_counting_features/outputs/extracted_datas_with_parc3.csv"

# Save with UTF-8 encoding and quotes (safer for long VEX/Decompiled strings)
#df.to_csv(output_csv, index=False, encoding="utf-8", quoting=1)  # quoting=1 = csv.QUOTE_ALL
#print(f"[+] Saved dataframe with parc3 features to: {output_csv}")

# Verify loading
df = load_data(output_csv)
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46


In [58]:
from typing import List, Optional, Dict, Any, Tuple
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [59]:
NUM_FEATURES_DEFAULT = [
    "switch_parc3",
    "switch_loop_parc3",
    "br_fact_parc3",
    "in_edges_parc3",
    "bb_cnt_parc3",
    "call_cnt_parc3",
]

In [60]:
def prepare_xy_by_program_split(
    df: pd.DataFrame,
    val_program: Optional[str] = None,
    features: Optional[List[str]] = None,
    label_col: str = "label",
    impute: Optional[str] = None,  # None, 'mean', 'median'
) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, LabelEncoder, str]:
    """Split df into training and validation by program name."""
    features = features or NUM_FEATURES_DEFAULT
    if "program" not in df.columns:
        raise ValueError("DataFrame must contain a 'program' column for the split.")

    if val_program is None:
        val_program = df["program"].value_counts().idxmax()

    mask_val = df["program"] == val_program
    mask_train = ~mask_val

    if mask_val.sum() == 0:
        raise ValueError(f"No rows found for val_program = '{val_program}'")

    X_train = df.loc[mask_train, features].copy()
    X_val = df.loc[mask_val, features].copy()
    y_train = df.loc[mask_train, label_col].copy()
    y_val = df.loc[mask_val, label_col].copy()

    if impute in {"mean", "median"}:
        imp = SimpleImputer(strategy=impute)
        X_train[:] = imp.fit_transform(X_train)
        X_val[:] = imp.transform(X_val)
    else:
        train_mask = X_train.notna().all(axis=1) & y_train.notna()
        val_mask = X_val.notna().all(axis=1) & y_val.notna()
        X_train = X_train.loc[train_mask]
        y_train = y_train.loc[train_mask]
        X_val = X_val.loc[val_mask]
        y_val = y_val.loc[val_mask]

    le = LabelEncoder()
    le.fit(pd.concat([y_train, y_val], axis=0))

    y_train_enc = le.transform(y_train)
    y_val_enc = le.transform(y_val)

    return X_train, X_val, y_train_enc, y_val_enc, le, val_program


def build_pipeline(xgb_params: Optional[Dict[str, Any]] = None) -> Pipeline:
    """Returns a Pipeline: StandardScaler -> XGBClassifier."""
    default = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    }
    if xgb_params:
        default.update(xgb_params)

    clf = XGBClassifier(**default)
    return Pipeline([("scaler", StandardScaler()), ("xgb", clf)])


def train_pipeline(pipeline: Pipeline, X_train: pd.DataFrame, y_train: np.ndarray) -> Pipeline:
    """Fit pipeline on training data and return fitted pipeline."""
    pipeline.fit(X_train, y_train)
    return pipeline


def evaluate_pipeline_only_recall(
    pipeline: Pipeline, X_val: pd.DataFrame, y_val_enc: np.ndarray, label_encoder: LabelEncoder
) -> Dict[str, Any]:
    """Evaluate pipeline on validation set and return recall metrics (macro + per-class)."""
    y_pred_enc = pipeline.predict(X_val)

    recall_macro = recall_score(y_val_enc, y_pred_enc, average="macro")
    recalls_per_class = recall_score(y_val_enc, y_pred_enc, average=None)

    classes_orig = label_encoder.classes_
    per_class_dict = {str(classes_orig[i]): float(recalls_per_class[i]) for i in range(len(recalls_per_class))}

    try:
        target_names_safe = [str(c) for c in label_encoder.classes_]
        report = classification_report(y_val_enc, y_pred_enc, target_names=target_names_safe, zero_division=0)
    except Exception:
        report = classification_report(y_val_enc, y_pred_enc, zero_division=0)

    cm = confusion_matrix(y_val_enc, y_pred_enc)

    return {
        "recall_macro": recall_macro,
        "recall_per_class": per_class_dict,
        "classification_report": report,
        "confusion_matrix": cm,
    }


def run_all(
    df: pd.DataFrame,
    val_program: Optional[str] = None,
    features: Optional[List[str]] = None,
    label_col: str = "label",
    impute: Optional[str] = None,
    xgb_params: Optional[Dict[str, Any]] = None,
    save_path: Optional[str] = "xgb_pipeline_by_program_only_recall.joblib",
) -> Dict[str, Any]:
    """End-to-end pipeline: split, build, train, evaluate, and optionally save."""
    features = features or NUM_FEATURES_DEFAULT

    X_train, X_val, y_train_enc, y_val_enc, le, val_prog_used = prepare_xy_by_program_split(
        df, val_program=val_program, features=features, label_col=label_col, impute=impute
    )

    pipeline = build_pipeline(xgb_params=xgb_params)
    pipeline = train_pipeline(pipeline, X_train, y_train_enc)

    metrics = evaluate_pipeline_only_recall(pipeline, X_val, y_val_enc, le)

    y_val_raw = le.inverse_transform(y_val_enc)

    if save_path:
        joblib.dump({"pipeline": pipeline, "label_encoder": le, "features": features}, save_path)

    return {
        "pipeline": pipeline,
        "metrics": metrics,
        "label_encoder": le,
        "val_program": val_prog_used,
        "X_val": X_val,
        "y_val_raw": pd.Series(y_val_raw, index=X_val.index),
        "features": features,
    }

In [61]:
df.program.unique()

array(['cparserXML', 'picohttpparser', 'csimpleJSONparser', 'cJSON',
       'benoitc_HTTP', 'yacc_calculator', 'network_packet_analyzer',
       'elf_parser', 'pcap_parser', 'packcc'], dtype=object)

In [62]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "cparserXML"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:56:50] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: cparserXML
Recall per class: {'0': 0.9493670886075949, '1': 0.29411764705882354}
Returned metrics: 0.6217423678332092
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [63]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "picohttpparser"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:56:51] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: picohttpparser
Recall per class: {'0': 0.9671361502347418, '1': 0.24550898203592814}
Returned metrics: 0.606322566135335
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [65]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "csimpleJSONparser"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:57:47] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: csimpleJSONparser
Recall per class: {'0': 0.8059701492537313, '1': 0.3695652173913043}
Returned metrics: 0.5877676833225178
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [66]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "cJSON"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: cJSON
Recall per class: {'0': 0.8726020659124447, '1': 0.6038461538461538}
Returned metrics: 0.7382241098792992
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [67]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "benoitc_HTTP"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:20] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: benoitc_HTTP
Recall per class: {'0': 0.8957055214723927, '1': 0.45263157894736844}
Returned metrics: 0.6741685502098805
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [68]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "yacc_calculator"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:28] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: yacc_calculator
Recall per class: {'0': 0.874439461883408, '1': 0.6551724137931034}
Returned metrics: 0.7648059378382557
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [69]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "network_packet_analyzer"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:37] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: network_packet_analyzer
Recall per class: {'0': 0.8926380368098159, '1': 0.0}
Returned metrics: 0.44631901840490795
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [70]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "elf_parser"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:44] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: elf_parser
Recall per class: {'0': 0.8111111111111111, '1': 0.2445414847161572}
Returned metrics: 0.5278262979136341
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [71]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "pcap_parser"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:58:52] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: pcap_parser
Recall per class: {'0': 0.9022556390977443, '1': 0.35714285714285715}
Returned metrics: 0.6296992481203008
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [72]:
# Start Pipeline of Feature Normalization + XGBoost + Recall Evaluation
val_program = "packcc"
results = run_all(df, val_program=val_program, impute=None)

# Accessing returned items:
val_program = results["val_program"]
metrics = results["metrics"]
recall_per_class = metrics["recall_per_class"]
features = results["features"]

print(f"Validation program: {val_program}")
print(f"Recall per class: {recall_per_class}")
print("Returned metrics:", metrics["recall_macro"])
print(f"Features used: {features}")

/home/marcos/anaconda3/envs/test-3.10.0-env/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [17:59:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Validation program: packcc
Recall per class: {'0': 0.6583601286173634, '1': 0.5931899641577061}
Returned metrics: 0.6257750463875347
Features used: ['switch_parc3', 'switch_loop_parc3', 'br_fact_parc3', 'in_edges_parc3', 'bb_cnt_parc3', 'call_cnt_parc3']


In [73]:
import numpy as np

values = [0.62, 0.60, 0.58, 0.73, 0.67, 0.76, 0.44, 0.52, 0.62, 0.62]

print(np.mean(values), np.std(values))

0.616 0.08856635930193811


## Feature Normalization + XGBoost with Hyperparameter Tuning

In [17]:
import numpy as np
import pandas as pd
import warnings
import joblib
from typing import Optional, List, Dict, Any, Tuple

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, classification_report, confusion_matrix, make_scorer
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from xgboost import XGBClassifier

In [18]:
# Feature list
NUM_FEATURES_DEFAULT = [
    "switch_parc3",
    "switch_loop_parc3",
    "br_fact_parc3",
    "in_edges_parc3",
    "bb_cnt_parc3",
    "call_cnt_parc3",
]


In [ ]:
# -----------------------------
# Split per program
# -----------------------------
def prepare_xy_by_program_split(
    df: pd.DataFrame,
    val_program: Optional[str] = None,
    features: Optional[List[str]] = None,
    label_col: str = "label",
    impute: Optional[str] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, LabelEncoder, str]:
    features = features or NUM_FEATURES_DEFAULT
    if "program" not in df.columns:
        raise ValueError("DataFrame must contain a 'program' column for the split.")

    if val_program is None:
        val_program = df["program"].value_counts().idxmax()

    mask_val = df["program"] == val_program
    mask_train = ~mask_val

    if mask_val.sum() == 0:
        raise ValueError(f"No rows found for val_program = '{val_program}'")

    X_train = df.loc[mask_train, features].copy()
    X_val = df.loc[mask_val, features].copy()
    y_train = df.loc[mask_train, label_col].copy()
    y_val = df.loc[mask_val, label_col].copy()

    if impute in {"mean", "median"}:
        imp = SimpleImputer(strategy=impute)
        X_train[:] = imp.fit_transform(X_train)
        X_val[:] = imp.transform(X_val)
    else:
        train_mask = X_train.notna().all(axis=1) & y_train.notna()
        val_mask = X_val.notna().all(axis=1) & y_val.notna()
        X_train = X_train.loc[train_mask]
        y_train = y_train.loc[train_mask]
        X_val = X_val.loc[val_mask]
        y_val = y_val.loc[val_mask]

    le = LabelEncoder()
    le.fit(pd.concat([y_train, y_val], axis=0))

    y_train_enc = le.transform(y_train)
    y_val_enc = le.transform(y_val)

    return X_train, X_val, y_train_enc, y_val_enc, le, val_program

# -----------------------------
# Pipeline building
# -----------------------------
def build_pipeline(xgb_params: Optional[Dict[str, Any]] = None) -> Pipeline:
    default = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
    }
    if xgb_params:
        default.update(xgb_params)

    clf = XGBClassifier(**default)
    return Pipeline([("scaler", StandardScaler()), ("xgb", clf)])

# -----------------------------
# Global Grid Search
# -----------------------------
def find_best_params_across_programs(df: pd.DataFrame, features: Optional[List[str]] = None, param_grid: dict = None) -> dict:
    features = features or NUM_FEATURES_DEFAULT
    X = df[features].values
    y = df["label"].values
    groups = df["program"].values

    scorer = make_scorer(recall_score, average="macro", zero_division=0)

    pipe = build_pipeline()

    if param_grid is None:
        param_grid = {
            "xgb__n_estimators": [100, 200],
            "xgb__max_depth": [3, 6],
            "xgb__learning_rate": [0.01, 0.1],
        }

    cv = LeaveOneGroupOut()
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=scorer,
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        grid.fit(X, y, groups=groups)

    print("Global best params:", grid.best_params_)
    return grid.best_params_

# -----------------------------
# Evaluation
# -----------------------------
def evaluate_pipeline_only_recall(
    pipeline: Pipeline, X_val: pd.DataFrame, y_val_enc: np.ndarray, label_encoder: LabelEncoder
) -> Dict[str, Any]:
    y_pred_enc = pipeline.predict(X_val)

    recall_macro = recall_score(y_val_enc, y_pred_enc, average="macro", zero_division=0)
    recalls_per_class = recall_score(y_val_enc, y_pred_enc, average=None, zero_division=0)

    classes_orig = label_encoder.classes_
    per_class_dict = {str(classes_orig[i]): float(recalls_per_class[i]) for i in range(len(recalls_per_class))}

    try:
        target_names_safe = [str(c) for c in label_encoder.classes_]
        report = classification_report(y_val_enc, y_pred_enc, target_names=target_names_safe, zero_division=0)
    except Exception:
        report = classification_report(y_val_enc, y_pred_enc, zero_division=0)

    cm = confusion_matrix(y_val_enc, y_pred_enc)

    return {
        "recall_macro": recall_macro,
        "recall_per_class": per_class_dict,
        "classification_report": report,
        "confusion_matrix": cm,
    }

# -----------------------------
# Leave-one-program-out with global best params
# -----------------------------
def leave_one_program_out_with_best_params(
    df: pd.DataFrame,
    best_params: dict,
    features: Optional[List[str]] = None,
    label_col: str = "label",
    impute: Optional[str] = None,
) -> pd.DataFrame:
    features = features or NUM_FEATURES_DEFAULT
    results = []

    for prog in df["program"].unique():
        X_train, X_val, y_train_enc, y_val_enc, le, val_prog_used = prepare_xy_by_program_split(
            df, val_program=prog, features=features, label_col=label_col, impute=impute
        )

        pipe = build_pipeline()
        pipe.set_params(**best_params)
        pipe.fit(X_train, y_train_enc)

        metrics = evaluate_pipeline_only_recall(pipe, X_val, y_val_enc, le)
        results.append({
            "program": prog,
            "recall_macro": metrics["recall_macro"],
            "recall_per_class": metrics["recall_per_class"]
        })

    return pd.DataFrame(results)

In [20]:
# Global Grid Search for best params
param_grid = {
    "xgb__n_estimators": [25, 50, 100],
    "xgb__max_depth": [3, 6, 10],
    "xgb__learning_rate": [0.01, 0.1],
}
# Find best params across all programs
best_params = find_best_params_across_programs(df, param_grid=param_grid)
# Leave-one-program-out evaluation with best params
results_df = leave_one_program_out_with_best_params(df, best_params)
print(results_df)
print("\nAverage Recall (macro): {:.4f} ± {:.4f}".format(
    results_df["recall_macro"].mean(), results_df["recall_macro"].std()
))

Fitting 10 folds for each of 18 candidates, totalling 180 fits
Global best params: {'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__n_estimators': 50}
                   program  recall_macro  \
0               cparserXML      0.590686   
1           picohttpparser      0.559030   
2        csimpleJSONparser      0.616929   
3                    cJSON      0.787306   
4             benoitc_HTTP      0.902680   
5          yacc_calculator      0.793954   
6  network_packet_analyzer      0.440184   
7               elf_parser      0.704682   
8              pcap_parser      0.501880   
9                   packcc      0.604126   

                                    recall_per_class  
0               {'0': 1.0, '1': 0.18137254901960784}  
1  {'0': 0.9953051643192489, '1': 0.1227544910179...  
2  {'0': 0.8805970149253731, '1': 0.3532608695652...  
3   {'0': 0.897688145597639, '1': 0.676923076923077}  
4  {'0': 0.8895705521472392, '1': 0.9157894736842...  
5  {'0': 0.9327354260089686,

## Manual Score from paper PIE + Replicate the experiments to find the right weights

$score = \sum_{f \in Features}{} w_{f} \frac{ x_{f} - min(X_{f}) }{max(X_{f}) - min(X_{f})}$

In [14]:
import math
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix

In [15]:
# ----------------- CONFIG -----------------
FEATURES = [
    "switch_loop_parc3",   # w0
    "br_fact_parc3",       # w1
    "in_edges_parc3",      # w2
    "bb_cnt_parc3",        # w3
    "call_cnt_parc3",      # w4
    "switch_parc3"         # w5
]
LABEL_COL = "label"   # binary: 1 => Parser, 0 => not
STEP = 0.1            # grid step for weights and threshold
K_PERCENT = 2.0       # top K percent to average
NORMALIZE_WEIGHTS = True  # require weights sum to 1
RANDOM_STATE = 42

In [16]:
def normalize_train(df_train, df_to_norm, features):
    mins = df_train[features].min()
    maxs = df_train[features].max()
    denom = (maxs - mins).replace(0, 1.0)
    norm = (df_to_norm[features] - mins) / denom
    return norm.clip(0, 1), mins, maxs

def compositions_of_N_into_k(N, k):
    for separators in combinations(range(N + k - 1), k - 1):
        parts, last = [], -1
        for s in separators:
            parts.append(s - last - 1)
            last = s
        parts.append((N + k - 1) - last - 1)
        yield tuple(parts)

def generate_weight_vectors(step=0.1, k=6, normalize=True):
    values = np.round(np.arange(0.0, 1.0 + 1e-12, step), 10)
    if not normalize:
        from itertools import product
        for comb in product(values, repeat=k):
            yield np.array(comb, dtype=float)
    else:
        N = int(round(1.0 / step))
        for parts in compositions_of_N_into_k(N, k):
            yield np.array(parts, dtype=float) / float(N)

def compute_fp_tp(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return fpr, tpr

def search_best_params(X_train_norm, y_train, step=0.1, normalize_weights=True):
    thresholds = np.round(np.arange(0.0, 1.0 + 1e-12, step), 10)
    results = []
    for w in generate_weight_vectors(step=step, k=X_train_norm.shape[1], normalize=normalize_weights):
        scores = X_train_norm.values.dot(w)
        for T in thresholds:
            preds = (scores > T).astype(int)
            fpr, tpr = compute_fp_tp(y_train.values, preds)
            dist = math.sqrt((fpr - 0.0)**2 + (tpr - 1.0)**2)
            results.append({
                "weights": w,
                "threshold": float(T),
                "fpr": fpr,
                "tpr": tpr,
                "dist": dist
            })
    return sorted(results, key=lambda r: r["dist"])

def average_top_k_percent(results_sorted, k_percent=2.0):
    n = len(results_sorted)
    top_k = max(1, int(round(n * (k_percent / 100.0))))
    top_subset = results_sorted[:top_k]
    weights_stack = np.vstack([r["weights"] for r in top_subset])
    avg_w = weights_stack.mean(axis=0)
    avg_T = np.mean([r["threshold"] for r in top_subset])
    return avg_w, float(avg_T), top_subset

In [17]:
def run_two_fold_cv(df, features=FEATURES, label_col=LABEL_COL,
                    step=STEP, normalize_weights=NORMALIZE_WEIGHTS,
                    k_percent=K_PERCENT, random_state=RANDOM_STATE):
    X = df[features].copy()
    y = df[label_col].copy()
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=random_state)
    splits = list(skf.split(X, y))
    _, test_idx = splits[0]
    S1_idx = test_idx
    S0_idx = np.setdiff1d(np.arange(len(X)), S1_idx)

    outputs = []
    for (train_idx, val_idx, direction) in [(S0_idx, S1_idx, "S0->S1"), (S1_idx, S0_idx, "S1->S0")]:
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_val = df.iloc[val_idx].reset_index(drop=True)
        Xtr_norm, _, _ = normalize_train(df_train, df_train, features)
        Xval_norm, _, _ = normalize_train(df_train, df_val, features)
        results_sorted = search_best_params(Xtr_norm, df_train[label_col],
                                            step=step, normalize_weights=normalize_weights)
        avg_w, avg_T, _ = average_top_k_percent(results_sorted, k_percent=k_percent)
        scores_val = Xval_norm.values.dot(avg_w)
        preds_val = (scores_val > avg_T).astype(int)
        fpr_val, tpr_val = compute_fp_tp(df_val[label_col].values, preds_val)
        d_val = math.sqrt((fpr_val - 0.0)**2 + (tpr_val - 1.0)**2)
        outputs.append({
            "direction": direction,
            "avg_weights": avg_w,
            "avg_threshold": avg_T,
            "fpr_val": fpr_val,
            "tpr_val": tpr_val,
            "d_val": d_val
        })
    eps = abs(outputs[0]["d_val"] - outputs[1]["d_val"])
    return outputs, eps

In [18]:
def pretty_print_results(outputs, eps):
    print("\n==== FINAL RESULTS ====")
    for out in outputs:
        w = out["avg_weights"]
        print(f"\nDirection {out['direction']}:")
        print("  avg threshold T = {:.4f}".format(out["avg_threshold"]))
        print("  avg weights:")
        for i, fname in enumerate(FEATURES):
            print(f"    w{i} ({fname}): {w[i]:.4f}")
        print("  Validation FPR = {:.4f}".format(out["fpr_val"]))
        print("  Validation TPR = {:.4f}".format(out["tpr_val"]))
        print("  Validation distance D = {:.4f}".format(out["d_val"]))
    print("\nEpsilon (|D_S0->S1 - D_S1->S0|) = {:.6f}".format(eps))
    print("========================\n")

In [19]:
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46


In [20]:
# run the procedure
outputs, eps = run_two_fold_cv(df)

# print results
pretty_print_results(outputs, eps)


==== FINAL RESULTS ====

Direction S0->S1:
  avg threshold T = 0.0917
  avg weights:
    w0 (switch_loop_parc3): 0.0778
    w1 (br_fact_parc3): 0.3915
    w2 (in_edges_parc3): 0.1319
    w3 (bb_cnt_parc3): 0.1067
    w4 (call_cnt_parc3): 0.2144
    w5 (switch_parc3): 0.0778
  Validation FPR = 0.2036
  Validation TPR = 0.2672
  Validation distance D = 0.7606

Direction S1->S0:
  avg threshold T = 0.0917
  avg weights:
    w0 (switch_loop_parc3): 0.0766
    w1 (br_fact_parc3): 0.3923
    w2 (in_edges_parc3): 0.1442
    w3 (bb_cnt_parc3): 0.1086
    w4 (call_cnt_parc3): 0.2017
    w5 (switch_parc3): 0.0767
  Validation FPR = 0.2037
  Validation TPR = 0.2113
  Validation distance D = 0.8146

Epsilon (|D_S0->S1 - D_S1->S0|) = 0.053996



In [21]:
# pick one direction (e.g., S0->S1) or average across directions if you want
best = outputs[0]   # or outputs[1], depending on which fold you want

# unpack into variables
w0, w1, w2, w3, w4, w5 = best["avg_weights"]
T = best["avg_threshold"]

print(f"Chosen weights: w0={w0}, w1={w1}, w2={w2}, w3={w3}, w4={w4}, w5={w5}")
print(f"Chosen threshold: T={T}")

Chosen weights: w0=0.07776096822995501, w1=0.39152798789712695, w2=0.1319213313161876, w3=0.10665658093797292, w4=0.21437216338880452, w5=0.07776096822995501
Chosen threshold: T=0.09167927382753403


In [22]:
def compute_parc3_score(df: pd.DataFrame, weights, threshold) -> pd.DataFrame:
    df = df.copy()
    score = 0
    
    for feature, w in weights.items():
        min_val = df[feature].min()
        max_val = df[feature].max()
        if max_val > min_val:
            norm = (df[feature] - min_val) / (max_val - min_val)
        else:
            norm = 0  # constant feature
        score += w * norm
    
    df["parc3_score"] = score
    df["parc3_pred"] = (df["parc3_score"] > threshold).astype(int)
    return df

In [23]:
# given weights
weights = {
    "switch_loop_parc3": w0,
    "br_fact_parc3": w1,
    "in_edges_parc3": w2,
    "bb_cnt_parc3": w3,
    "call_cnt_parc3": w4,
    "switch_parc3": w5,
}

T = best["avg_threshold"] # threshold

df = compute_parc3_score(df, weights=weights, threshold=T)
df.head()

,name,address,label,num_blocks,success,error_message,vex_ir,decompiled_c,program,option,compiler,switch_parc3,switch_loop_parc3,br_fact_parc3,in_edges_parc3,bb_cnt_parc3,call_cnt_parc3,parc3_score,parc3_pred
0,sym.deregister_tm_clones,0x401110,0,4,True,NaN,--- BLOCK 0x401110 ---\nIRSB {\n t0:Ity_I64 ...,long long deregister_tm_clones()\n{\n if (t...,cparserXML,default,gcc,0,0,2,6,4,384,0.181856,1
1,sym.register_tm_clones,0x401140,0,4,True,NaN,--- BLOCK 0x401140 ---\nIRSB {\n t0:Ity_I64 ...,long long register_tm_clones()\n{\n if (tru...,cparserXML,default,gcc,0,0,2,11,4,259,0.138095,1
2,sym.__do_global_dtors_aux,0x401180,0,4,True,NaN,--- BLOCK 0x401180 ---\nIRSB {\n t0:Ity_I8 t...,extern char __bss_start;\n\nlong long __do_glo...,cparserXML,default,gcc,0,0,2,5,4,216,0.122560,1
3,sym.frame_dummy,0x4011b0,0,1,True,NaN,--- BLOCK 0x4011b0 ---\nIRSB {\n t0:Ity_I64\...,long long frame_dummy()\n{\n return registe...,cparserXML,default,gcc,0,0,0,2,1,121,0.042663,0
4,sym.skip_whitespace,0x4011cb,1,14,True,NaN,--- BLOCK 0x40120a ---\nIRSB {\n t0:Ity_I64 ...,long long skip_whitespace(unsigned long a0)\n{...,cparserXML,default,gcc,0,0,2,6,14,46,0.063190,0


In [24]:
def compute_recall_per_file(df: pd.DataFrame) -> pd.DataFrame:
    recalls = []
    
    for file, group in df.groupby("program"):
        tp = ((group["label"] == 1) & (group["parc3_pred"] == 1)).sum()
        fn = ((group["label"] == 1) & (group["parc3_pred"] == 0)).sum()
        
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        
        recalls.append({"file_name": file, "recall": recall})
    
    return pd.DataFrame(recalls)

In [25]:
recall_df = compute_recall_per_file(df)
print(recall_df)

                 file_name    recall
0             benoitc_HTTP  0.084211
1                    cJSON  0.103846
2               cparserXML  0.122549
3        csimpleJSONparser  0.244565
4               elf_parser  0.183406
5  network_packet_analyzer  0.000000
6                   packcc  0.413978
7              pcap_parser  0.000000
8           picohttpparser  0.338323
9          yacc_calculator  0.068966


In [26]:
import numpy as np

values = [0.084211, 0.103846, 0.122549, 0.244565, 0.183406, 0.000000, 0.413978, 0.000000, 0.338323, 0.068966]

print(np.mean(values), np.std(values))

0.1559844 0.13176826523651286
